# Laboratorio 3: Conciliacion de movimientos

Aprenderemos a comparar dos tablas mediante una clave acordada. Usaremos un libro sintetico y una cartola sintetica, sin decidir reglas contables por nuestra cuenta.

## Objetivos

- Construir una clave de conciliacion.
- Usar `merge` e `indicator`.
- Separar coincidencias, faltantes y diferencias.
- Visualizar pendientes para orientar la revision humana.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'laboratorios' else Path.cwd()
EJEMPLOS = ROOT / 'ejemplos'
libro = pd.read_excel(EJEMPLOS / 'practica_conciliacion_banco.xlsx')
libro

## 1. Separar las fuentes

En un caso real estas tablas vendrian de sistemas distintos. Aqui el ejemplo trae montos de libro y banco para mostrar el procedimiento.

In [ ]:
libro_contable = libro[['Fecha', 'Referencia', 'Descripcion', 'Monto_Libro']].copy()
banco = libro[['Fecha', 'Referencia', 'Descripcion', 'Monto_Banco']].copy()
libro_contable['clave'] = libro_contable['Referencia']
banco['clave'] = banco['Referencia']
libro_contable

## 2. Conciliar con merge

La referencia es una clave didactica. Antes de usar datos reales, la persona responsable debe aprobar si la clave correcta sera referencia, documento, fecha+monto u otra combinacion.

In [ ]:
conciliacion = libro_contable.merge(
    banco[['clave', 'Monto_Banco']],
    on='clave',
    how='outer',
    indicator=True,
    validate='one_to_one',
)
conciliacion['diferencia'] = conciliacion['Monto_Libro'].fillna(0) - conciliacion['Monto_Banco'].fillna(0)
conciliacion

## 3. Clasificar resultados

Una coincidencia no implica aprobacion automatica. La diferencia debe compararse con una tolerancia previamente acordada.

In [ ]:
tolerancia = 1000
conciliacion['estado'] = 'PENDIENTE'
conciliacion.loc[(conciliacion['_merge'] == 'both') & (conciliacion['diferencia'].abs() <= tolerancia), 'estado'] = 'COINCIDE_TOLERANCIA'
conciliacion.loc[conciliacion['_merge'] != 'both', 'estado'] = 'MOVIMIENTO_FALTANTE'
conciliacion.loc[(conciliacion['_merge'] == 'both') & (conciliacion['diferencia'].abs() > tolerancia), 'estado'] = 'DIFERENCIA_SIGNIFICATIVA'
conciliacion[['clave', 'Monto_Libro', 'Monto_Banco', 'diferencia', 'estado']]

In [ ]:
resumen = conciliacion['estado'].value_counts()
ax = resumen.plot(kind='bar', figsize=(8, 4), title='Estados de conciliacion')
ax.set_ylabel('Movimientos')
plt.tight_layout()
plt.show()

## Ejercicio de cierre

1. Agrega una fila solo al libro y observa el estado.
2. Cambia la tolerancia y observa que cambia.
3. Documenta que movimientos requieren revisar referencia, fecha o monto.
4. No apruebes una diferencia solo porque esta dentro de la tolerancia: registra el criterio y el responsable.